# Train and evaluate models

Single end-to-end pipeline. Each section runs in order and produces
results saved into `results/predictions.pkl` for notebook 04 to consume ;p yum

**Flow**
1. Setup + data
2. Logistic regression baseline
3. XGBoost: strongest tabular baseline (DLs greatest op)
4. MLP initial training
5. Supervised autoencoder (SAE) initial training
6. Hyperparameter sweep (MLP + SAE) selects the best config
7. Retrain MLP and SAE at the sweep-winning configs
8. Further SAE experiments (none of these improved model oops)
9. Final comparison table + save predictions and checkpoints

In [1]:
# colab setup
import sys, os, subprocess

IN_COLAB = 'google.colab' in sys.modules
PROJECT_DIR_DRIVE = '/content/drive/MyDrive/final_project'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_DIR = PROJECT_DIR_DRIVE
    subprocess.run(['pip', 'install', '-q', 'scanpy', 'umap-learn', 'xgboost'], check=True)
else:
    PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

DATA_DIR = os.path.join(PROJECT_DIR, 'data')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'models'), exist_ok=True)
print('PROJECT_DIR =', PROJECT_DIR)

Mounted at /content/drive
PROJECT_DIR = /content/drive/MyDrive/final_project


### Step 1. Imports and data

In [2]:
import importlib
import pickle
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

from src import data as D
from src import train as T
from src import evaluate as E

T.set_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
## RUN IN T4 HIGH RAM ELSE TAKES FIVEVER!!
adata = ad.read_h5ad(Path(DATA_DIR) / 'processed.h5ad')
print(adata)

device: cuda
AnnData object with n_obs × n_vars = 95790 × 2000
    obs: 'sample', 'UMAP_1', 'UMAP_2', 'ident', 'patient', 'source', 'clonotype', 'clone_size', 'expanded', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'split'
    var: 'gene_id', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p'
    layers: 'counts'


In [3]:
def split_arrays(adata, split):
    sub = adata[adata.obs['split'] == split]
    return D.to_matrix(sub), sub.obs['expanded'].values.astype(np.float32)

X_train, y_train = split_arrays(adata, 'train')
X_val,   y_val   = split_arrays(adata, 'val')
X_test,  y_test  = split_arrays(adata, 'test')
gene_names = list(adata.var_names)

print('train:', X_train.shape, 'pos frac', y_train.mean())
print('val:  ', X_val.shape,   'pos frac', y_val.mean())
print('test: ', X_test.shape,  'pos frac', y_test.mean())

train: (65435, 2000) pos frac 0.4967372
val:   (14224, 2000) pos frac 0.5857002
test:  (16131, 2000) pos frac 0.49060816


## Step 2: logistic regression baseline
L2-regularized logistic regression with `class_weight='balanced'`.  Reference point for how much non-linearity buys.

In [4]:
logreg = LogisticRegression(max_iter=1000, n_jobs=-1, class_weight='balanced', C=1.0)
logreg.fit(X_train, y_train)
p_logreg_val  = logreg.predict_proba(X_val)[:, 1]
p_logreg_test = logreg.predict_proba(X_test)[:, 1]
print('val :', E.classification_metrics(y_val,  p_logreg_val))
print('test:', E.classification_metrics(y_test, p_logreg_test))

val : {'auroc': 0.8002726084871725, 'auprc': 0.8148083829040322, 'f1': 0.7749662203660483, 'positive_rate': 0.5857002139091492, 'threshold': 0.5}
test: {'auroc': 0.7881968904558124, 'auprc': 0.7510177433180918, 'f1': 0.7252283034105734, 'positive_rate': 0.4906081557273865, 'threshold': 0.5}


## Step 3. XGBoost (strongest tabular baseline)

Gradient-boosted trees, no preprocessing changes... Often the best  model for moderate-size tabular biology data.

In [5]:
xgb = XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    n_jobs=-1, eval_metric='auc', tree_method='hist',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
p_xgb_val  = xgb.predict_proba(X_val)[:, 1]
p_xgb_test = xgb.predict_proba(X_test)[:, 1]
print('val :', E.classification_metrics(y_val,  p_xgb_val))
print('test:', E.classification_metrics(y_test, p_xgb_test))

top_xgb = (pd.Series(xgb.feature_importances_, index=gene_names)
             .sort_values(ascending=False).head(30))
top_xgb

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [20:32:47] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


val : {'auroc': 0.8311931583164685, 'auprc': 0.8601306089152051, 'f1': 0.8020914718432598, 'positive_rate': 0.5857002139091492, 'threshold': 0.5}
test: {'auroc': 0.8108133285933189, 'auprc': 0.7658293111905711, 'f1': 0.7499844556363863, 'positive_rate': 0.4906081557273865, 'threshold': 0.5}


,0
NKG7,0.089185
EEF1A1,0.017778
GZMH,0.017651
CCL4,0.011454
SELL,0.011081
CCR7,0.008637
XCL1,0.008249
CXCR6,0.007850
PASK,0.006506
HOPX,0.006465


## Step 4. MLP initial training

Default config (lr=1e-3, hidden=(512,128), dropout=0.3). Sweep in step 5 will tune this; the result here is the untuned baseline

In [6]:
cfg = T.TrainConfig(epochs=20, batch_size=512, lr=1e-3)
mlp_init, mlp_init_hist = T.train_mlp(
    X_train, y_train, X_val, y_val,
    in_features=X_train.shape[1], cfg=cfg,
)
p_mlp_init_val  = T.predict_proba(mlp_init, X_val)
p_mlp_init_test = T.predict_proba(mlp_init, X_test)
print('val :', E.classification_metrics(y_val,  p_mlp_init_val))
print('test:', E.classification_metrics(y_test, p_mlp_init_test))

epoch   1/20  train=0.4911  val=0.5141
epoch   2/20  train=0.4477  val=0.5031
epoch   3/20  train=0.4268  val=0.5077
epoch   4/20  train=0.4000  val=0.5383
epoch   5/20  train=0.3693  val=0.5511
epoch   6/20  train=0.3367  val=0.6012
epoch   7/20  train=0.2955  val=0.5914
epoch   8/20  train=0.2590  val=0.6818
epoch   9/20  train=0.2278  val=0.7343
epoch  10/20  train=0.1992  val=0.7053
epoch  11/20  train=0.1740  val=0.8403
epoch  12/20  train=0.1512  val=0.8683
epoch  13/20  train=0.1343  val=0.8924
epoch  14/20  train=0.1212  val=0.9480
epoch  15/20  train=0.1083  val=0.9593
epoch  16/20  train=0.0988  val=0.9842
epoch  17/20  train=0.0916  val=1.0998
epoch  18/20  train=0.0806  val=1.1304
epoch  19/20  train=0.0767  val=1.1477
epoch  20/20  train=0.0734  val=1.1652
val : {'auroc': 0.7811345805707323, 'auprc': 0.8006281162518601, 'f1': 0.7507034327518289, 'positive_rate': 0.5857002139091492, 'threshold': 0.5}
test: {'auroc': 0.7708652746857119, 'auprc': 0.7377973267917484, 'f1': 0.7

## Step 5. Supervised autoencoder initial training

Joint loss = `recon_weight · MSE(decoder(z), x) + clf_weight · BCE(logit, y)`. Starting point: equal-ish weights, latent_dim=32.

In [7]:
cfg = T.TrainConfig(epochs=30, batch_size=512, lr=1e-3,
                    recon_weight=1.0, clf_weight=2.0)
sae_init, sae_init_hist = T.train_sae(
    X_train, y_train, X_val, y_val,
    in_features=X_train.shape[1], cfg=cfg,
    hidden_dims=(512, 128), latent_dim=32, dropout=0.2,
)
p_sae_init_val  = T.predict_proba(sae_init, X_val)
p_sae_init_test = T.predict_proba(sae_init, X_test)
print('val :', E.classification_metrics(y_val,  p_sae_init_val))
print('test:', E.classification_metrics(y_test, p_sae_init_test))

epoch   1/30  train: loss=1.1076 recon=0.1088 clf=0.4994  val: loss=1.1394 recon=0.1011 clf=0.5192
epoch   2/30  train: loss=0.9970 recon=0.0900 clf=0.4535  val: loss=1.1424 recon=0.0961 clf=0.5232
epoch   3/30  train: loss=0.9489 recon=0.0861 clf=0.4314  val: loss=1.1103 recon=0.0940 clf=0.5081
epoch   4/30  train: loss=0.8950 recon=0.0843 clf=0.4054  val: loss=1.1473 recon=0.0928 clf=0.5272
epoch   5/30  train: loss=0.8289 recon=0.0831 clf=0.3729  val: loss=1.1695 recon=0.0917 clf=0.5389
epoch   6/30  train: loss=0.7470 recon=0.0824 clf=0.3323  val: loss=1.2443 recon=0.0919 clf=0.5762
epoch   7/30  train: loss=0.6658 recon=0.0821 clf=0.2919  val: loss=1.3664 recon=0.0912 clf=0.6376
epoch   8/30  train: loss=0.5836 recon=0.0820 clf=0.2508  val: loss=1.6660 recon=0.0913 clf=0.7873
epoch   9/30  train: loss=0.5154 recon=0.0822 clf=0.2166  val: loss=1.6657 recon=0.0920 clf=0.7868
epoch  10/30  train: loss=0.4590 recon=0.0822 clf=0.1884  val: loss=1.9861 recon=0.0913 clf=0.9474
epoch  11/

## STEP 6. Hyperparameter sweep

Small grid/model. Best config is selected on **val AUROC**  only, test predictions in step 6 use the winning config retrained from scratch

In [8]:
records = []

# MLP grid
mlp_grid = [
    dict(lr=1e-3, hidden_dims=(512, 128), dropout=0.3),   # og
    dict(lr=3e-4, hidden_dims=(512, 128), dropout=0.3),
    dict(lr=3e-4, hidden_dims=(256,),     dropout=0.1),
    dict(lr=1e-4, hidden_dims=(256, 64),  dropout=0.1),
]
for i, hp in enumerate(mlp_grid):
    cfg = T.TrainConfig(epochs=25, batch_size=512, lr=hp['lr'], weight_decay=1e-4)
    print(f'\nMLP run {i+1}/{len(mlp_grid)}: {hp}')
    m, _ = T.train_mlp(X_train, y_train, X_val, y_val,
                       in_features=X_train.shape[1], cfg=cfg,
                       hidden_dims=hp['hidden_dims'], dropout=hp['dropout'])
    p_val = T.predict_proba(m, X_val)
    records.append({'model': 'mlp', **hp,
                    'val_auroc': roc_auc_score(y_val, p_val),
                    'val_auprc': average_precision_score(y_val, p_val)})

# SAE grid
sae_grid = [
    dict(lr=1e-3, recon_w=1.0, clf_w=2.0,  latent=32, hidden=(512,128), dropout=0.2),
    dict(lr=3e-4, recon_w=0.1, clf_w=10.0, latent=32, hidden=(512,128), dropout=0.2),
    dict(lr=3e-4, recon_w=0.5, clf_w=5.0,  latent=16, hidden=(256, 64), dropout=0.1),
    dict(lr=3e-4, recon_w=1.0, clf_w=1.0,  latent=64, hidden=(512,128), dropout=0.2),
]
for i, hp in enumerate(sae_grid):
    cfg = T.TrainConfig(epochs=30, batch_size=512, lr=hp['lr'], weight_decay=1e-4,
                        recon_weight=hp['recon_w'], clf_weight=hp['clf_w'])
    print(f'\nSAE run {i+1}/{len(sae_grid)}: {hp}')
    m, _ = T.train_sae(X_train, y_train, X_val, y_val,
                       in_features=X_train.shape[1], cfg=cfg,
                       hidden_dims=hp['hidden'], latent_dim=hp['latent'],
                       dropout=hp['dropout'])
    p_val = T.predict_proba(m, X_val)
    records.append({'model': 'sae', **hp,
                    'val_auroc': roc_auc_score(y_val, p_val),
                    'val_auprc': average_precision_score(y_val, p_val)})

sweep_df = (pd.DataFrame(records)
              .sort_values(['model', 'val_auroc'], ascending=[True, False])
              .reset_index(drop=True))
sweep_df.to_csv(Path(RESULTS_DIR) / 'hp_sweep.csv', index=False)
sweep_df


MLP run 1/4: {'lr': 0.001, 'hidden_dims': (512, 128), 'dropout': 0.3}
epoch   1/25  train=0.4913  val=0.5141
epoch   2/25  train=0.4493  val=0.5072
epoch   3/25  train=0.4312  val=0.5186
epoch   4/25  train=0.4101  val=0.5228
epoch   5/25  train=0.3869  val=0.5212
epoch   6/25  train=0.3624  val=0.5564
epoch   7/25  train=0.3324  val=0.5577
epoch   8/25  train=0.3007  val=0.6138
epoch   9/25  train=0.2732  val=0.6454
epoch  10/25  train=0.2504  val=0.6426
epoch  11/25  train=0.2225  val=0.7056
epoch  12/25  train=0.1991  val=0.7649
epoch  13/25  train=0.1826  val=0.7587
epoch  14/25  train=0.1686  val=0.7913
epoch  15/25  train=0.1546  val=0.8280
epoch  16/25  train=0.1402  val=0.9658
epoch  17/25  train=0.1333  val=0.9624
epoch  18/25  train=0.1239  val=0.9571
epoch  19/25  train=0.1149  val=0.9651
epoch  20/25  train=0.1083  val=0.9996
epoch  21/25  train=0.1013  val=0.9871
epoch  22/25  train=0.0968  val=1.0157
epoch  23/25  train=0.0977  val=1.0754
epoch  24/25  train=0.0908  val=

,model,lr,hidden_dims,dropout,val_auroc,val_auprc,recon_w,clf_w,latent,hidden
0,mlp,0.0003,"(256,)",0.1,0.817006,0.846849,NaN,NaN,NaN,NaN
1,mlp,0.0001,"(256, 64)",0.1,0.810398,0.840240,NaN,NaN,NaN,NaN
2,mlp,0.0003,"(512, 128)",0.3,0.791686,0.811650,NaN,NaN,NaN,NaN
3,mlp,0.0010,"(512, 128)",0.3,0.787243,0.805505,NaN,NaN,NaN,NaN
4,sae,0.0003,NaN,0.2,0.786774,0.803028,1.0,1.0,64.0,"(512, 128)"
5,sae,0.0003,NaN,0.2,0.783418,0.800203,0.1,10.0,32.0,"(512, 128)"
6,sae,0.0010,NaN,0.2,0.774356,0.795833,1.0,2.0,32.0,"(512, 128)"
7,sae,0.0003,NaN,0.1,0.755449,0.769520,0.5,5.0,16.0,"(256, 64)"


## Step 7. Retrain MLP and SAE at the sweep-winning configs

These models go into the final results BUNDLE! Test set is only touched here, never used to select configs!!!

In [9]:
best_mlp = sweep_df[sweep_df.model == 'mlp'].iloc[0]
best_sae = sweep_df[sweep_df.model == 'sae'].iloc[0]
print('best MLP:', dict(best_mlp))
print('best SAE:', dict(best_sae))

best MLP: {'model': 'mlp', 'lr': np.float64(0.0003), 'hidden_dims': (256,), 'dropout': np.float64(0.1), 'val_auroc': np.float64(0.817005533991398), 'val_auprc': np.float64(0.8468488002654398), 'recon_w': np.float64(nan), 'clf_w': np.float64(nan), 'latent': np.float64(nan), 'hidden': nan}
best SAE: {'model': 'sae', 'lr': np.float64(0.0003), 'hidden_dims': nan, 'dropout': np.float64(0.2), 'val_auroc': np.float64(0.7867739094555503), 'val_auprc': np.float64(0.8030278978920249), 'recon_w': np.float64(1.0), 'clf_w': np.float64(1.0), 'latent': np.float64(64.0), 'hidden': (512, 128)}


In [10]:
# BEST MLP
cfg = T.TrainConfig(epochs=25, batch_size=512, lr=best_mlp['lr'], weight_decay=1e-4)
mlp, mlp_hist = T.train_mlp(
    X_train, y_train, X_val, y_val,
    in_features=X_train.shape[1], cfg=cfg,
    hidden_dims=tuple(best_mlp['hidden_dims']),
    dropout=float(best_mlp['dropout']),
)
p_mlp_val  = T.predict_proba(mlp, X_val)
p_mlp_test = T.predict_proba(mlp, X_test)
print('val :', E.classification_metrics(y_val,  p_mlp_val))
print('test:', E.classification_metrics(y_test, p_mlp_test))

epoch   1/25  train=0.5182  val=0.5301
epoch   2/25  train=0.4729  val=0.5165
epoch   3/25  train=0.4591  val=0.5208
epoch   4/25  train=0.4486  val=0.5076
epoch   5/25  train=0.4380  val=0.5041
epoch   6/25  train=0.4285  val=0.5039
epoch   7/25  train=0.4189  val=0.5094
epoch   8/25  train=0.4105  val=0.5019
epoch   9/25  train=0.4010  val=0.4979
epoch  10/25  train=0.3927  val=0.4998
epoch  11/25  train=0.3829  val=0.5151
epoch  12/25  train=0.3732  val=0.5154
epoch  13/25  train=0.3623  val=0.5150
epoch  14/25  train=0.3530  val=0.5300
epoch  15/25  train=0.3421  val=0.5196
epoch  16/25  train=0.3315  val=0.5177
epoch  17/25  train=0.3196  val=0.5237
epoch  18/25  train=0.3085  val=0.5356
epoch  19/25  train=0.2979  val=0.5434
epoch  20/25  train=0.2855  val=0.5454
epoch  21/25  train=0.2746  val=0.5366
epoch  22/25  train=0.2647  val=0.5421
epoch  23/25  train=0.2525  val=0.5622
epoch  24/25  train=0.2425  val=0.5581
epoch  25/25  train=0.2330  val=0.5961
val : {'auroc': 0.8170055

In [11]:
# BEST SAE
cfg = T.TrainConfig(epochs=30, batch_size=512, lr=best_sae['lr'], weight_decay=1e-4,
                    recon_weight=best_sae['recon_w'], clf_weight=best_sae['clf_w'])
sae, sae_hist = T.train_sae(
    X_train, y_train, X_val, y_val,
    in_features=X_train.shape[1], cfg=cfg,
    hidden_dims=tuple(best_sae['hidden']),
    latent_dim=int(best_sae['latent']),
    dropout=float(best_sae['dropout']),
)
p_sae_val  = T.predict_proba(sae, X_val)
p_sae_test = T.predict_proba(sae, X_test)
print('val :', E.classification_metrics(y_val,  p_sae_val))
print('test:', E.classification_metrics(y_test, p_sae_test))

# save for l8r (architecture metadata)
BEST_SAE_HIDDEN  = tuple(best_sae['hidden'])
BEST_SAE_LATENT  = int(best_sae['latent'])
BEST_SAE_DROPOUT = float(best_sae['dropout'])
BEST_MLP_HIDDEN  = tuple(best_mlp['hidden_dims'])
BEST_MLP_DROPOUT = float(best_mlp['dropout'])
print('saved config metadata:')
print('  MLP hidden:', BEST_MLP_HIDDEN, 'dropout:', BEST_MLP_DROPOUT)
print('  SAE hidden:', BEST_SAE_HIDDEN, 'latent:', BEST_SAE_LATENT, 'dropout:', BEST_SAE_DROPOUT)

epoch   1/30  train: loss=0.6466 recon=0.1242 clf=0.5224  val: loss=0.6216 recon=0.1027 clf=0.5189
epoch   2/30  train: loss=0.5577 recon=0.0929 clf=0.4648  val: loss=0.6187 recon=0.1003 clf=0.5184
epoch   3/30  train: loss=0.5351 recon=0.0909 clf=0.4443  val: loss=0.6102 recon=0.0993 clf=0.5109
epoch   4/30  train: loss=0.5114 recon=0.0884 clf=0.4230  val: loss=0.6042 recon=0.0981 clf=0.5062
epoch   5/30  train: loss=0.4861 recon=0.0872 clf=0.3989  val: loss=0.6185 recon=0.0977 clf=0.5208
epoch   6/30  train: loss=0.4562 recon=0.0870 clf=0.3693  val: loss=0.6491 recon=0.0975 clf=0.5515
epoch   7/30  train: loss=0.4177 recon=0.0868 clf=0.3309  val: loss=0.6521 recon=0.0976 clf=0.5545
epoch   8/30  train: loss=0.3746 recon=0.0866 clf=0.2880  val: loss=0.7214 recon=0.0975 clf=0.6239
epoch   9/30  train: loss=0.3339 recon=0.0865 clf=0.2474  val: loss=0.7665 recon=0.0986 clf=0.6678
epoch  10/30  train: loss=0.2927 recon=0.0863 clf=0.2064  val: loss=0.8546 recon=0.0979 clf=0.7567
epoch  11/

## Step. 8 Further SAE experiments (spoiler: they all fail lol)

1. **Aggressive classification weight** (`clf_weight=10`, smaller latent)
2. **Two-stage training** (pretrain reconstruction, then add classifier)
3. **Large latent** (`latent=128`, `clf_weight=20`)

In [17]:
extra_records = [
    {'name': 'MLP best', 'val_auroc': roc_auc_score(y_val, p_mlp_val),
     'val_auprc': average_precision_score(y_val, p_mlp_val)},
    {'name': 'SAE best', 'val_auroc': roc_auc_score(y_val, p_sae_val),
     'val_auprc': average_precision_score(y_val, p_sae_val)},
]

# AGGRESSIVE CLASSIFICATION WEIGHT
print('\n SAE w/ aggressive clf weight')
cfg = T.TrainConfig(epochs=40, batch_size=512, lr=3e-4, weight_decay=1e-4,
                    recon_weight=0.1, clf_weight=10.0)
sae_agg, _ = T.train_sae(
    X_train, y_train, X_val, y_val,
    in_features=X_train.shape[1], cfg=cfg,
    hidden_dims=(256, 64), latent_dim=16, dropout=0.1,
)
p_agg_val = T.predict_proba(sae_agg, X_val)
extra_records.append({'name': 'SAE aggressive-clf',
                      'val_auroc': roc_auc_score(y_val, p_agg_val),
                      'val_auprc': average_precision_score(y_val, p_agg_val)})


 SAE w/ aggressive clf weight
epoch   1/40  train: loss=5.5318 recon=0.1400 clf=0.5518  val: loss=5.3367 recon=0.1036 clf=0.5326
epoch   2/40  train: loss=4.7991 recon=0.0955 clf=0.4790  val: loss=5.3122 recon=0.1037 clf=0.5302
epoch   3/40  train: loss=4.6621 recon=0.0949 clf=0.4653  val: loss=5.1696 recon=0.1047 clf=0.5159
epoch   4/40  train: loss=4.5092 recon=0.0954 clf=0.4500  val: loss=5.0499 recon=0.1056 clf=0.5039
epoch   5/40  train: loss=4.3615 recon=0.0959 clf=0.4352  val: loss=5.0836 recon=0.1065 clf=0.5073
epoch   6/40  train: loss=4.2135 recon=0.0966 clf=0.4204  val: loss=5.3413 recon=0.1082 clf=0.5330
epoch   7/40  train: loss=4.0322 recon=0.0972 clf=0.4022  val: loss=5.2353 recon=0.1092 clf=0.5224
epoch   8/40  train: loss=3.8294 recon=0.0975 clf=0.3820  val: loss=5.4679 recon=0.1091 clf=0.5457
epoch   9/40  train: loss=3.6148 recon=0.0979 clf=0.3605  val: loss=5.3938 recon=0.1086 clf=0.5383
epoch  10/40  train: loss=3.3405 recon=0.0980 clf=0.3331  val: loss=5.6590 rec

In [18]:
# TWO STAGE TRAINING
print('\n Two-stage SAE (pretrain recon, then classifier)')
cfg1 = T.TrainConfig(epochs=20, batch_size=512, lr=1e-3,
                    recon_weight=1.0, clf_weight=0.0)
sae_pre, _ = T.train_sae(
    X_train, y_train, X_val, y_val,
    in_features=X_train.shape[1], cfg=cfg1,
    hidden_dims=(512, 128), latent_dim=32, dropout=0.2,
)
cfg2 = T.TrainConfig(epochs=20, batch_size=512, lr=3e-4)
sae_two, _ = T.train_sae_finetune(
    sae_pre, X_train, y_train, X_val, y_val,
    cfg=cfg2, freeze_encoder=False,
)
p_two_val = T.predict_proba(sae_two, X_val)
extra_records.append({'name': 'SAE two-stage',
                      'val_auroc': roc_auc_score(y_val, p_two_val),
                      'val_auprc': average_precision_score(y_val, p_two_val)})


 Two-stage SAE (pretrain recon, then classifier)
epoch   1/20  train: loss=0.1042 recon=0.1042 clf=0.7009  val: loss=0.0932 recon=0.0932 clf=0.7001
epoch   2/20  train: loss=0.0810 recon=0.0810 clf=0.6982  val: loss=0.0871 recon=0.0871 clf=0.6986
epoch   3/20  train: loss=0.0765 recon=0.0765 clf=0.6977  val: loss=0.0835 recon=0.0835 clf=0.6985
epoch   4/20  train: loss=0.0744 recon=0.0744 clf=0.6977  val: loss=0.0821 recon=0.0821 clf=0.6985
epoch   5/20  train: loss=0.0732 recon=0.0732 clf=0.6977  val: loss=0.0812 recon=0.0812 clf=0.6985
epoch   6/20  train: loss=0.0723 recon=0.0723 clf=0.6977  val: loss=0.0804 recon=0.0804 clf=0.6985
epoch   7/20  train: loss=0.0715 recon=0.0715 clf=0.6977  val: loss=0.0794 recon=0.0794 clf=0.6985
epoch   8/20  train: loss=0.0710 recon=0.0710 clf=0.6977  val: loss=0.0790 recon=0.0790 clf=0.6985
epoch   9/20  train: loss=0.0705 recon=0.0705 clf=0.6977  val: loss=0.0785 recon=0.0785 clf=0.6985
epoch  10/20  train: loss=0.0702 recon=0.0702 clf=0.6977  v

In [19]:
# LARGE LATENT & CLASSIFICATION WEIGHT
print('\nSAE w/ large latent, very strong clf weigh')
cfg = T.TrainConfig(epochs=40, batch_size=512, lr=3e-4, weight_decay=1e-4,
                    recon_weight=0.05, clf_weight=20.0)
sae_big, _ = T.train_sae(
    X_train, y_train, X_val, y_val,
    in_features=X_train.shape[1], cfg=cfg,
    hidden_dims=(512, 128), latent_dim=128, dropout=0.1,
)
p_big_val = T.predict_proba(sae_big, X_val)
extra_records.append({'name': 'SAE large-latent',
                      'val_auroc': roc_auc_score(y_val, p_big_val),
                      'val_auprc': average_precision_score(y_val, p_big_val)})

extras_df = pd.DataFrame(extra_records).sort_values('val_auroc', ascending=False)
extras_df.to_csv(Path(RESULTS_DIR) / 'sae_extra_experiments.csv', index=False)
extras_df


SAE w/ large latent, very strong clf weigh
epoch   1/40  train: loss=10.3197 recon=0.1286 clf=0.5157  val: loss=10.4313 recon=0.1057 clf=0.5213
epoch   2/40  train: loss=9.2321 recon=0.0966 clf=0.4614  val: loss=10.2002 recon=0.1095 clf=0.5097
epoch   3/40  train: loss=8.7320 recon=0.0977 clf=0.4364  val: loss=10.1132 recon=0.1085 clf=0.5054
epoch   4/40  train: loss=8.0697 recon=0.0987 clf=0.4032  val: loss=10.2551 recon=0.1133 clf=0.5125
epoch   5/40  train: loss=7.1727 recon=0.0994 clf=0.3584  val: loss=10.7154 recon=0.1138 clf=0.5355
epoch   6/40  train: loss=6.0109 recon=0.1004 clf=0.3003  val: loss=11.5965 recon=0.1137 clf=0.5795
epoch   7/40  train: loss=4.7669 recon=0.1010 clf=0.2381  val: loss=14.3705 recon=0.1137 clf=0.7182
epoch   8/40  train: loss=3.7372 recon=0.1014 clf=0.1866  val: loss=15.0857 recon=0.1157 clf=0.7540
epoch   9/40  train: loss=2.8964 recon=0.1014 clf=0.1446  val: loss=19.0105 recon=0.1150 clf=0.9502
epoch  10/40  train: loss=2.2263 recon=0.1015 clf=0.111

,name,val_auroc,val_auprc
0,MLP best,0.817006,0.846849
4,SAE large-latent,0.789664,0.810860
1,SAE best,0.786774,0.803028
3,SAE two-stage,0.780776,0.811694
2,SAE aggressive-clf,0.777199,0.800493


## Step 9. Final comparison + save predictions and checkpoints
Everything below is consumed by `04_evaluation_interpretation.ipynb` (yum ;p)

In [20]:
summary = []
for name, p_val, p_test in [
    ('logreg', p_logreg_val, p_logreg_test),
    ('mlp',    p_mlp_val,    p_mlp_test),
    ('sae',    p_sae_val,    p_sae_test),
    ('xgb',    p_xgb_val,    p_xgb_test),
]:
    summary.append({
        'model': name,
        'val_auroc':  roc_auc_score(y_val,  p_val),
        'val_auprc':  average_precision_score(y_val, p_val),
        'test_auroc': roc_auc_score(y_test, p_test),
        'test_auprc': average_precision_score(y_test, p_test),
    })
summary_df = pd.DataFrame(summary).set_index('model')
summary_df.to_csv(Path(RESULTS_DIR) / 'final_comparison.csv')
summary_df

,val_auroc,val_auprc,test_auroc,test_auprc
model,,,,
logreg,0.800273,0.814808,0.788197,0.751018
mlp,0.817006,0.846849,0.775779,0.738264
sae,0.786774,0.803028,0.771420,0.739853
xgb,0.831193,0.860131,0.810813,0.765829


In [21]:
preds = {
    'gene_names': gene_names,
    'y_val':  y_val,
    'y_test': y_test,
    'logreg': {'val': p_logreg_val, 'test': p_logreg_test,
               'coef': logreg.coef_.ravel()},
    'mlp':    {'val': p_mlp_val,    'test': p_mlp_test,
               'history': mlp_hist,
               'config': {'hidden_dims': BEST_MLP_HIDDEN,
                          'dropout': BEST_MLP_DROPOUT}},
    'sae':    {'val': p_sae_val,    'test': p_sae_test,
               'history': sae_hist,
               'config': {'hidden_dims': BEST_SAE_HIDDEN,
                          'latent_dim': BEST_SAE_LATENT,
                          'dropout': BEST_SAE_DROPOUT}},
    'xgb':    {'val': p_xgb_val,    'test': p_xgb_test,
               'feature_importance': xgb.feature_importances_},
}
with open(Path(RESULTS_DIR) / 'predictions.pkl', 'wb') as f:
    pickle.dump(preds, f)

torch.save(mlp.state_dict(), Path(RESULTS_DIR) / 'models' / 'mlp.pt')
torch.save(sae.state_dict(), Path(RESULTS_DIR) / 'models' / 'sae.pt')

print('saved:')
print('  results/predictions.pkl')
print('  results/models/mlp.pt')
print('  results/models/sae.pt')
print('  results/final_comparison.csv')
print('  results/hp_sweep.csv')
print('  results/sae_extra_experiments.csv')

saved:
  results/predictions.pkl
  results/models/mlp.pt
  results/models/sae.pt
  results/final_comparison.csv
  results/hp_sweep.csv
  results/sae_extra_experiments.csv
